# Groundwater Prediction Pipeline (Pune)

This notebook trains a Spark time-series regression pipeline on the cleaned Pune groundwater dataset and generates future groundwater forecasts.

Workflow:
1. Load filtered data
2. Run quality checks and validation
3. Build lag and rainfall-driven time-series features
4. Train and validate a Spark ML model
5. Forecast future groundwater levels
6. Save forecasts into a Gold Delta table

In [1]:
import os
import math
from datetime import timedelta

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("GroundWaterPrediction-Pipeline")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

PROJECT_ROOT = os.getcwd()
INPUT_PATH = os.path.join(PROJECT_ROOT, "data", "Maharashtra_Pune_Filtered.csv")
GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "gold", "groundwater_prediction_gold")
GOLD_TABLE = "bhujal_mitra.groundwater_prediction_gold"
FORECAST_HORIZON_DAYS = 30

print("Input path:", INPUT_PATH)
print("Gold path:", GOLD_PATH)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 01:11:20 WARN Utils: Your hostname, mridulUbuntu, resolves to a loopback address: 127.0.1.1; using 10.51.26.78 instead (on interface wlp2s0)
26/03/29 01:11:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/mridul-sharma/Desktop/bhujal-mitra/.venv-1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/mridul-sharma/.ivy2.5.2/cache
The jars for the packages stored in: /home/mridul-sharma/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-89bf9d09-7c62-4b24-a528-f90b4d904efb;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central


	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
:: resolution report :: resolve 271ms :: artifacts dl 9ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	io.delta#delta-spark_4.1_2.13;4.1.0 from central in [default]
	io.delta#delta-storage;4.1.0 from central in [default]
	io.unitycatalog#unitycatalog-client;0.4.0 from central in [default]
	org.apache.logging.log4j#log4j-api;2.25.3 from central in [default]
	org.apache.logging.log4j#log4j-core;2.25.3 from central in [default]
	org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 from central in [default]
	org.slf4j#slf4j-api;2.0.13 from central in [default]
	:: evicted modules:
	org.slf4j#slf4j-api;2.0.17 by [org.slf4j#slf4j-api;2.0.13] in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   

26/03/29 01:11:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Input path: /home/mridul-sharma/Desktop/bhujal-mitra/data/Maharashtra_Pune_Filtered.csv
Gold path: /home/mridul-sharma/Desktop/bhujal-mitra/data/gold/groundwater_prediction_gold


In [2]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)

df = (
    raw_df
    .withColumn("station_id", F.col("station_id").cast("string"))
    .withColumn("datetime", F.to_date("datetime"))
    .withColumn("groundwater_level_target", F.col("groundwater_level_target").cast("double"))
    .withColumn("rainfall", F.coalesce(F.col("rainfall").cast("double"), F.lit(0.0)))
    .filter(F.col("datetime").isNotNull() & F.col("groundwater_level_target").isNotNull())
)

print(f"Rows after load: {df.count()}")
print(f"Stations: {df.select('station_id').distinct().count()}")
print("Date range:")
df.select(F.min("datetime").alias("start"), F.max("datetime").alias("end")).show(truncate=False)

df.printSchema()
df.show(5, truncate=False)

Rows after load: 14641


Stations: 94
Date range:


+----------+----------+
|start     |end       |
+----------+----------+
|1994-01-05|2025-09-25|
+----------+----------+

root
 |-- station_id: string (nullable = true)
 |-- time_idx: integer (nullable = true)
 |-- datetime: date (nullable = true)
 |-- groundwater_level_target: double (nullable = true)
 |-- rainfall: double (nullable = false)
 |-- t2m_avg: double (nullable = true)
 |-- t2m_max: double (nullable = true)
 |-- t2m_min: double (nullable = true)
 |-- month: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- wellDepth: double (nullable = true)
 |-- wellAquiferType_encoded: integer (nullable = true)

+----------+--------+----------+------------------------+--------+-------+-------+-------+-----+--------+---------+---------+-----------------------+
|station_id|time_idx|datetime  |groundwater_level_target|rainfall|t2m_avg|t2m_max|t2m_min|month|latitude|longitude|wellDepth|wellAquiferType_encoded|
+----------+--------+-

In [3]:
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["station_id", "datetime", "groundwater_level_target", "rainfall"]]
null_summary = df.select(*null_exprs)
print("Null summary:")
null_summary.show(truncate=False)

duplicate_station_date = df.groupBy("station_id", "datetime").count().filter(F.col("count") > 1)
dup_count = duplicate_station_date.count()
print(f"Duplicate station-date keys: {dup_count}")

coverage_df = (
    df.groupBy("station_id")
    .agg(
        F.count("*").alias("records"),
        F.min("datetime").alias("start_date"),
        F.max("datetime").alias("end_date")
    )
    .orderBy(F.desc("records"))
)

print("Top 10 stations by history length:")
coverage_df.show(10, truncate=False)

print("Coverage summary:")
coverage_df.select(
    F.min("records").alias("min_records"),
    F.max("records").alias("max_records"),
    F.avg("records").alias("avg_records")
).show(truncate=False)

Null summary:


+----------+--------+------------------------+--------+
|station_id|datetime|groundwater_level_target|rainfall|
+----------+--------+------------------------+--------+
|0         |0       |0                       |0       |
+----------+--------+------------------------+--------+



Duplicate station-date keys: 0
Top 10 stations by history length:


+----------+-------+----------+----------+
|station_id|records|start_date|end_date  |
+----------+-------+----------+----------+
|CGWBNG0549|409    |2023-04-19|2025-01-06|
|CGWBNG0541|402    |2023-05-22|2025-09-06|
|CGWBNG0596|317    |2023-06-17|2025-06-28|
|CGWBNG0579|305    |2023-07-19|2025-09-14|
|CGWJPR0202|297    |2024-05-24|2025-08-25|
|CGWBNG0583|295    |2023-05-04|2024-11-23|
|CGWBNG0587|292    |2023-06-05|2025-06-03|
|CGWJPR0246|286    |2024-07-01|2025-09-25|
|CGWBNG0555|281    |2023-05-01|2025-08-21|
|CGWNAG2007|276    |2024-05-17|2025-07-26|
+----------+-------+----------+----------+
only showing top 10 rows
Coverage summary:


+-----------+-----------+------------------+
|min_records|max_records|avg_records       |
+-----------+-----------+------------------+
|106        |409        |155.75531914893617|
+-----------+-----------+------------------+



In [4]:
daily_df = (
    df.groupBy("station_id", "datetime")
    .agg(
        F.avg("groundwater_level_target").alias("groundwater_level_target"),
        F.avg("rainfall").alias("rainfall")
    )
    .orderBy("station_id", "datetime")
)

w = Window.partitionBy("station_id").orderBy("datetime")
w_roll = w.rowsBetween(-7, -1)

feature_df = (
    daily_df
    .withColumn("lag_1", F.lag("groundwater_level_target", 1).over(w))
    .withColumn("lag_3", F.lag("groundwater_level_target", 3).over(w))
    .withColumn("lag_7", F.lag("groundwater_level_target", 7).over(w))
    .withColumn("rain_lag_1", F.lag("rainfall", 1).over(w))
    .withColumn("rain_lag_3", F.lag("rainfall", 3).over(w))
    .withColumn("roll7_level", F.avg("groundwater_level_target").over(w_roll))
    .withColumn("roll7_rain", F.avg("rainfall").over(w_roll))
    .withColumn("month", F.month("datetime"))
    .withColumn("month_sin", F.sin(F.lit(2.0 * math.pi) * F.col("month") / F.lit(12.0)))
    .withColumn("month_cos", F.cos(F.lit(2.0 * math.pi) * F.col("month") / F.lit(12.0)))
    .withColumn("date_ord", F.unix_date("datetime"))
)

feature_cols = [
    "lag_1",
    "lag_3",
    "lag_7",
    "rainfall",
    "rain_lag_1",
    "rain_lag_3",
    "roll7_level",
    "roll7_rain",
    "month_sin",
    "month_cos"
]

model_df = feature_df.dropna(subset=feature_cols + ["groundwater_level_target"])

print(f"Rows available for modeling: {model_df.count()}")
model_df.select(F.min("datetime").alias("model_start"), F.max("datetime").alias("model_end")).show(truncate=False)
model_df.select("station_id", "datetime", "groundwater_level_target", *feature_cols).show(5, truncate=False)

Rows available for modeling: 13983


+-----------+----------+
|model_start|model_end |
+-----------+----------+
|1995-11-06 |2025-09-25|
+-----------+----------+



+---------------+----------+------------------------+-----+-----+-----+--------+----------+----------+-----------------+------------------+-------------------+-------------------+
|station_id     |datetime  |groundwater_level_target|lag_1|lag_3|lag_7|rainfall|rain_lag_1|rain_lag_3|roll7_level      |roll7_rain        |month_sin          |month_cos          |
+---------------+----------+------------------------+-----+-----+-----+--------+----------+----------+-----------------+------------------+-------------------+-------------------+
|153145076380001|1996-01-05|4.45                    |3.78 |6.99 |5.76 |0.0     |6.15      |8.45      |4.687142857142857|2.5385714285714287|0.49999999999999994|0.8660254037844387 |
|153145076380001|1996-01-25|4.45                    |4.45 |4.7  |5.24 |0.0     |0.0       |1.19      |4.5              |2.5342857142857143|0.49999999999999994|0.8660254037844387 |
|153145076380001|1996-05-15|5.87                    |4.45 |3.78 |3.0  |0.0     |0.0       |6.15     

In [5]:
split_ord = model_df.approxQuantile("date_ord", [0.8], 0.0)[0]

train_df = model_df.filter(F.col("date_ord") <= split_ord).cache()
valid_df = model_df.filter(F.col("date_ord") > split_ord).cache()

print(f"Train rows: {train_df.count()}")
print(f"Validation rows: {valid_df.count()}")

print("Train period:")
train_df.select(F.min("datetime").alias("start"), F.max("datetime").alias("end")).show(truncate=False)

print("Validation period:")
valid_df.select(F.min("datetime").alias("start"), F.max("datetime").alias("end")).show(truncate=False)

Train rows: 11196


Validation rows: 2787
Train period:


+----------+----------+
|start     |end       |
+----------+----------+
|1995-11-06|2024-08-15|
+----------+----------+

Validation period:


+----------+----------+
|start     |end       |
+----------+----------+
|2024-08-16|2025-09-25|
+----------+----------+



In [6]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
regressor = GBTRegressor(
    featuresCol="features",
    labelCol="groundwater_level_target",
    predictionCol="prediction",
    maxIter=120,
    maxDepth=5,
    stepSize=0.05,
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, regressor])
model = pipeline.fit(train_df)
valid_pred = model.transform(valid_df).cache()

rmse_eval = RegressionEvaluator(labelCol="groundwater_level_target", predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="groundwater_level_target", predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="groundwater_level_target", predictionCol="prediction", metricName="r2")

rmse = rmse_eval.evaluate(valid_pred)
mae = mae_eval.evaluate(valid_pred)
r2 = r2_eval.evaluate(valid_pred)

baseline_pred = valid_df.withColumn("baseline_prediction", F.col("lag_1"))
baseline_rmse = RegressionEvaluator(labelCol="groundwater_level_target", predictionCol="baseline_prediction", metricName="rmse").evaluate(baseline_pred)
baseline_mae = RegressionEvaluator(labelCol="groundwater_level_target", predictionCol="baseline_prediction", metricName="mae").evaluate(baseline_pred)

print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE: {mae:.4f}")
print(f"Validation R2: {r2:.4f}")
print(f"Naive baseline RMSE (lag_1): {baseline_rmse:.4f}")
print(f"Naive baseline MAE (lag_1): {baseline_mae:.4f}")


[Stage 176:====================================================>(198 + 2) / 200]




[Stage 181:=================================================>  (189 + 11) / 200]




[Stage 1551:==============================================>    (181 + 14) / 200]




[Stage 1566:===============================================>   (188 + 12) / 200]




[Stage 1741:=============================================>     (178 + 13) / 200]



...


[Stage 3146:===============================================>   (188 + 12) / 200]




[Stage 3156:==============================================>    (183 + 12) / 200]




[Stage 3161:==========================================>        (166 + 15) / 200]




[Stage 3171:==================================>                (137 + 12) / 200]



Validation RMSE: 12.0935
Validation MAE: 2.6143
Validation R2: 0.5808
Naive baseline RMSE (lag_1): 4.2425
Naive baseline MAE (lag_1): 0.3748


In [7]:
residual_df = (
    valid_pred
    .withColumn("error", F.col("prediction") - F.col("groundwater_level_target"))
    .withColumn("abs_error", F.abs(F.col("error")))
    .withColumn("sq_error", F.pow(F.col("error"), 2))
)

print("Residual summary:")
residual_df.select(
    F.avg("error").alias("mean_error"),
    F.avg("abs_error").alias("mean_abs_error"),
    F.sqrt(F.avg("sq_error")).alias("root_mean_sq_error")
).show(truncate=False)

print("Top 10 stations by validation MAE:")
residual_df.groupBy("station_id").agg(
    F.avg("abs_error").alias("station_mae"),
    F.count("*").alias("rows")
).orderBy(F.desc("station_mae")).show(10, truncate=False)

print("Sample predictions vs actuals:")
residual_df.select(
    "station_id",
    "datetime",
    "groundwater_level_target",
    "prediction",
    "error",
    "abs_error"
).orderBy(F.desc("abs_error")).show(15, truncate=False)

Residual summary:


+------------------+------------------+------------------+
|mean_error        |mean_abs_error    |root_mean_sq_error|
+------------------+------------------+------------------+
|-0.593804239698247|2.6143221924253486|12.093514729907708|
+------------------+------------------+------------------+

Top 10 stations by validation MAE:


+----------+------------------+----+
|station_id|station_mae       |rows|
+----------+------------------+----+
|CGWJPR0266|49.73987589048581 |40  |
|CGWNAG2189|29.826352900379337|100 |
|CGWBNG0555|2.4452511737993774|108 |
|CGWBNG0596|1.518620426252393 |145 |
|CGWBNG0579|1.500595634139391 |191 |
|CGWNAG2167|1.3318988361550688|182 |
|CGWBNG0498|1.292619781838111 |107 |
|CGWNAG2181|1.1414699904006773|2   |
|CGWNAG2202|0.9392021768014117|105 |
|CGWBNG0512|0.8949920092396944|66  |
+----------+------------------+----+
only showing top 10 rows
Sample predictions vs actuals:


+----------+----------+------------------------+-------------------+-------------------+------------------+
|station_id|datetime  |groundwater_level_target|prediction         |error              |abs_error         |
+----------+----------+------------------------+-------------------+-------------------+------------------+
|CGWJPR0266|2025-01-17|-144.56333333333333     |-2.3234591905745794|142.23987414275877 |142.23987414275877|
|CGWJPR0266|2025-01-14|-145.2                  |-9.502652678392456 |135.69734732160754 |135.69734732160754|
|CGWJPR0266|2025-04-24|1.0                     |-133.80416812794002|-134.80416812794002|134.80416812794002|
|CGWJPR0266|2025-01-13|-142.88333333333333     |-11.811298991023202|131.07203434231013 |131.07203434231013|
|CGWNAG2189|2024-11-15|-48.709999999999994     |-177.84470312128215|-129.13470312128214|129.13470312128214|
|CGWJPR0266|2025-04-27|-151.42333333333332     |-23.24002017095004 |128.18331316238329 |128.18331316238329|
|CGWNAG2189|2024-11-12|-48.6

In [8]:
final_model = pipeline.fit(model_df)
print("Final model trained on full historical feature set.")

station_monthly_rain = (
    daily_df
    .withColumn("month", F.month("datetime"))
    .groupBy("station_id", "month")
    .agg(F.avg("rainfall").alias("avg_monthly_rainfall"))
)

global_monthly_rain = (
    daily_df
    .withColumn("month", F.month("datetime"))
    .groupBy("month")
    .agg(F.avg("rainfall").alias("global_avg_monthly_rainfall"))
)

station_monthly_pd = station_monthly_rain.toPandas()
global_monthly_pd = global_monthly_rain.toPandas()

station_monthly_lookup = {
    (row.station_id, int(row.month)): float(row.avg_monthly_rainfall)
    for row in station_monthly_pd.itertuples(index=False)
}
global_monthly_lookup = {
    int(row.month): float(row.global_avg_monthly_rainfall)
    for row in global_monthly_pd.itertuples(index=False)
}

history_pd = (
    daily_df
    .select("station_id", "datetime", "groundwater_level_target", "rainfall")
    .orderBy("station_id", "datetime")
    .toPandas()
)
history_pd["datetime"] = pd.to_datetime(history_pd["datetime"])

station_histories = {}
for station_id, group in history_pd.groupby("station_id"):
    group = group.sort_values("datetime")
    station_histories[station_id] = [
        {
            "datetime": row.datetime.date(),
            "groundwater_level_target": float(row.groundwater_level_target),
            "rainfall": float(row.rainfall),
        }
        for row in group.itertuples(index=False)
    ]

print(f"Stations prepared for forecasting: {len(station_histories)}")

Final model trained on full historical feature set.


Stations prepared for forecasting: 94


In [9]:
future_records = []

for step in range(1, FORECAST_HORIZON_DAYS + 1):
    step_rows = []

    for station_id, history in station_histories.items():
        next_date = history[-1]["datetime"] + timedelta(days=1)
        month_value = next_date.month

        assumed_rainfall = station_monthly_lookup.get(
            (station_id, month_value),
            global_monthly_lookup.get(month_value, 0.0)
        )

        target_series = [h["groundwater_level_target"] for h in history]
        rain_series = [h["rainfall"] for h in history]

        lag_1 = target_series[-1]
        lag_3 = target_series[-3] if len(target_series) >= 3 else target_series[-1]
        lag_7 = target_series[-7] if len(target_series) >= 7 else float(np.mean(target_series))
        rain_lag_1 = rain_series[-1]
        rain_lag_3 = rain_series[-3] if len(rain_series) >= 3 else rain_series[-1]
        roll7_level = float(np.mean(target_series[-7:]))
        roll7_rain = float(np.mean(rain_series[-7:]))
        month_sin = math.sin((2.0 * math.pi * month_value) / 12.0)
        month_cos = math.cos((2.0 * math.pi * month_value) / 12.0)
        date_ord = int((pd.Timestamp(next_date) - pd.Timestamp("1970-01-01")) // pd.Timedelta("1D"))

        step_rows.append({
            "station_id": station_id,
            "forecast_date": next_date.isoformat(),
            "step_ahead_days": step,
            "assumed_rainfall": float(assumed_rainfall),
            "rainfall": float(assumed_rainfall),
            "lag_1": float(lag_1),
            "lag_3": float(lag_3),
            "lag_7": float(lag_7),
            "rain_lag_1": float(rain_lag_1),
            "rain_lag_3": float(rain_lag_3),
            "roll7_level": float(roll7_level),
            "roll7_rain": float(roll7_rain),
            "month_sin": float(month_sin),
            "month_cos": float(month_cos),
            "date_ord": int(date_ord),
        })

    step_df = spark.createDataFrame(pd.DataFrame(step_rows))
    step_predictions = (
        final_model.transform(step_df)
        .withColumn("forecast_date", F.to_date("forecast_date"))
        .select("station_id", "forecast_date", "step_ahead_days", "assumed_rainfall", "prediction")
    )

    step_pd = step_predictions.toPandas()

    for row in step_pd.itertuples(index=False):
        station_id = row.station_id
        forecast_date = pd.to_datetime(row.forecast_date).date()
        predicted_level = float(row.prediction)
        assumed_rainfall = float(row.assumed_rainfall)

        station_histories[station_id].append({
            "datetime": forecast_date,
            "groundwater_level_target": predicted_level,
            "rainfall": assumed_rainfall
        })

        future_records.append({
            "station_id": station_id,
            "forecast_date": forecast_date.isoformat(),
            "step_ahead_days": int(row.step_ahead_days),
            "assumed_rainfall": assumed_rainfall,
            "predicted_groundwater_level": predicted_level
        })

future_predictions_df = (
    spark.createDataFrame(pd.DataFrame(future_records))
    .withColumn("forecast_date", F.to_date("forecast_date"))
)

print(f"Future forecast rows generated: {future_predictions_df.count()}")
future_predictions_df.show(10, truncate=False)

Future forecast rows generated: 2820
+---------------+-------------+---------------+--------------------+---------------------------+
|station_id     |forecast_date|step_ahead_days|assumed_rainfall    |predicted_groundwater_level|
+---------------+-------------+---------------+--------------------+---------------------------+
|153145076380001|2023-05-26   |1              |2.8757142857142854  |5.323836702810319          |
|154000075340001|2024-01-06   |1              |0.036333333333333336|5.120662311852044          |
|154800075450001|2024-01-06   |1              |0.005172413793103449|8.806471531819613          |
|155015074223001|2023-05-11   |1              |1.1078260869565215  |12.192489358591379         |
|160324074195001|2023-05-11   |1              |1.0755555555555556  |2.7343970014556467         |
|161400074030001|2023-05-11   |1              |1.2967999999999997  |9.321345848633484          |
|162430073594501|2023-05-11   |1              |0.8585714285714288  |6.568156064183892     

In [10]:
gold_df = (
    future_predictions_df
    .withColumn("model_name", F.lit("spark_gbt_lag_rainfall_v1"))
    .withColumn("generated_at", F.current_timestamp())
)

spark.sql("CREATE DATABASE IF NOT EXISTS bhujal_mitra")

gold_df.write.format("delta").mode("overwrite").save(GOLD_PATH)
spark.sql(f"DROP TABLE IF EXISTS {GOLD_TABLE}")
spark.sql(f"CREATE TABLE {GOLD_TABLE} USING DELTA LOCATION '{GOLD_PATH}'")

print("Gold Delta table saved.")
print("Table name:", GOLD_TABLE)
print("Location:", GOLD_PATH)


[Stage 6344:====================================================> (49 + 1) / 50]



Gold Delta table saved.
Table name: bhujal_mitra.groundwater_prediction_gold
Location: /home/mridul-sharma/Desktop/bhujal-mitra/data/gold/groundwater_prediction_gold


In [11]:
gold_loaded = spark.read.format("delta").load(GOLD_PATH)

print(f"Gold rows: {gold_loaded.count()}")
print(f"Stations in Gold: {gold_loaded.select('station_id').distinct().count()}")

print("Per-station horizon validation:")
gold_loaded.groupBy("station_id").count().select(
    F.min("count").alias("min_forecast_days"),
    F.max("count").alias("max_forecast_days"),
    F.avg("count").alias("avg_forecast_days")
).show(truncate=False)

print("Prediction value range check:")
gold_loaded.select(
    F.min("predicted_groundwater_level").alias("min_predicted_level"),
    F.max("predicted_groundwater_level").alias("max_predicted_level"),
    F.avg("predicted_groundwater_level").alias("avg_predicted_level")
).show(truncate=False)

print("Null check in Gold:")
gold_loaded.select(
    F.sum(F.col("predicted_groundwater_level").isNull().cast("int")).alias("null_predictions"),
    F.sum(F.col("assumed_rainfall").isNull().cast("int")).alias("null_assumed_rainfall")
).show(truncate=False)

gold_loaded.orderBy("station_id", "forecast_date").show(20, truncate=False)

Gold rows: 2820


Stations in Gold: 94
Per-station horizon validation:


+-----------------+-----------------+-----------------+
|min_forecast_days|max_forecast_days|avg_forecast_days|
+-----------------+-----------------+-----------------+
|30               |30               |30.0             |
+-----------------+-----------------+-----------------+

Prediction value range check:


+-------------------+-------------------+-------------------+
|min_predicted_level|max_predicted_level|avg_predicted_level|
+-------------------+-------------------+-------------------+
|-76.77416150529848 |29.423555802658754 |3.818649491815016  |
+-------------------+-------------------+-------------------+

Null check in Gold:


+----------------+---------------------+
|null_predictions|null_assumed_rainfall|
+----------------+---------------------+
|0               |0                    |
+----------------+---------------------+

+---------------+-------------+---------------+------------------+---------------------------+-------------------------+--------------------------+
|station_id     |forecast_date|step_ahead_days|assumed_rainfall  |predicted_groundwater_level|model_name               |generated_at              |
+---------------+-------------+---------------+------------------+---------------------------+-------------------------+--------------------------+
|153145076380001|2023-05-26   |1              |2.8757142857142854|5.323836702810319          |spark_gbt_lag_rainfall_v1|2026-03-29 01:18:11.373566|
|153145076380001|2023-05-27   |2              |2.8757142857142854|6.5398785073322845         |spark_gbt_lag_rainfall_v1|2026-03-29 01:18:11.373566|
|153145076380001|2023-05-28   |3              |2.87571

In [12]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
